# Phase 3 — Feature engineering

# Setting up libraries

In [1]:
import pandas as pd
import numpy as np

import datetime as dt
from datetime import date

from dateutil.relativedelta import relativedelta

In [2]:
print("table: online retail transactions")
retail_data = pd.read_csv('../data/interim/cleaned_retail_transactions.csv')
display(retail_data.head())

table: online retail transactions


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,is_cancellation,is_non_product,is_missing_customer,is_customer_cancellation,is_stock_adjustment
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,False,False,False,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,False,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,False,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,False,False,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,False,False,False,False


In [3]:
retail_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1028761 entries, 0 to 1028760
Data columns (total 13 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   invoice                   1028761 non-null  str    
 1   stockcode                 1028761 non-null  str    
 2   description               1028761 non-null  str    
 3   quantity                  1028761 non-null  int64  
 4   invoicedate               1028761 non-null  str    
 5   price                     1028761 non-null  float64
 6   customer_id               797885 non-null   float64
 7   country                   1028761 non-null  str    
 8   is_cancellation           1028761 non-null  bool   
 9   is_non_product            1028761 non-null  bool   
 10  is_missing_customer       1028761 non-null  bool   
 11  is_customer_cancellation  1028761 non-null  bool   
 12  is_stock_adjustment       1028761 non-null  bool   
dtypes: bool(5), float64(2), int64(1), str(

In [4]:
retail_data["customer_id"] = retail_data["customer_id"].astype("Int64")

In [5]:
retail_data["invoicedate"] = pd.to_datetime(retail_data["invoicedate"], errors='coerce')

In [6]:
retail_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1028761 entries, 0 to 1028760
Data columns (total 13 columns):
 #   Column                    Non-Null Count    Dtype         
---  ------                    --------------    -----         
 0   invoice                   1028761 non-null  str           
 1   stockcode                 1028761 non-null  str           
 2   description               1028761 non-null  str           
 3   quantity                  1028761 non-null  int64         
 4   invoicedate               1028761 non-null  datetime64[us]
 5   price                     1028761 non-null  float64       
 6   customer_id               797885 non-null   Int64         
 7   country                   1028761 non-null  str           
 8   is_cancellation           1028761 non-null  bool          
 9   is_non_product            1028761 non-null  bool          
 10  is_missing_customer       1028761 non-null  bool          
 11  is_customer_cancellation  1028761 non-null  bool          
 1

In [7]:
# Revenue calculation
retail_data["revenue"] = retail_data["quantity"] * retail_data["price"]

In [ ]:
# Extract year component
retail_data["invoice_year"] = retail_data["invoicedate"].dt.year

# Extract month component
retail_data["invoice_month"] = retail_data["invoicedate"].dt.month

# Extract year-month component
retail_data["year_month"] = retail_data["invoicedate"].dt.to_period("M")

# Extract quarter component
retail_data["invoice_quarter"] = retail_data["invoicedate"].dt.quarter

# Extract weekend indicator
retail_data["is_weekend"] = retail_data["invoicedate"].dt.dayofweek >= 5

# Extract day of month component
retail_data["invoice_day"] = retail_data["invoicedate"].dt.day

# Extract hour component
retail_data["invoice_hour"] = retail_data["invoicedate"].dt.hour

# Extract day of week component
retail_data["invoice_weekday"] = retail_data["invoicedate"].dt.weekday

In [11]:
# Create a new column to indicate if the quantity is negative
retail_data["is_negative_quantity"] = (retail_data["quantity"] < 0)

In [12]:
# Create a new column to indicate valid sales
retail_data["is_valid_sale"] = ((retail_data["is_cancellation"] == False) &
    (retail_data["quantity"] > 0) &
    (retail_data["price"] > 0))

In [ ]:
order_summary = (
    retail_data[retail_data["is_valid_sale"] == True]
    .groupby("invoice")
    .agg(
        order_revenue=("revenue", "sum"),
        order_units=("quantity", "sum"),
        order_product_count=("stockcode", "nunique"),
        order_line_count=("stockcode", "size")
    )
    .reset_index()
)